## Contexto da Análise Exploratória — Perdas Não-Técnicas

A concessionária de energia do Ceará chegou com dois problemas aparentemente separados: perdas não-técnicas crescentes (a conta não fecha, energia sai da rede mas não é faturada) e um aumento inexplicável de quedas de energia concentrado em um bairro. A hipótese inicial da engenharia era que os dois fenômenos estavam conectados, o furto estaria sobrecarregando a infraestrutura local e causando os apagões.

Com 5.000 unidades consumidoras, 50 transformadores e 580 registros de ocorrências cobrindo 2023 e 2024, o objetivo era localizar o problema, identificar o transformador crítico e estimar o prejuízo financeiro.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
import warnings
warnings.filterwarnings('ignore')

# ══════════════════════════════════════════════════════════════════
# 1. CARREGAMENTO DOS DADOS
# ══════════════════════════════════════════════════════════════════
df_consumo = pd.read_csv('../data/projeto_06_consumo_energia.csv')
df_transf  = pd.read_csv('../data/projeto_06_cadastro_transformadores.csv')
df_oc      = pd.read_csv('../data/projeto_06_ocorrencias_rede.csv')

df_oc['data_ocorrencia'] = pd.to_datetime(df_oc['data_ocorrencia'])
df_oc['periodo'] = df_oc['data_ocorrencia'].dt.to_period('M').dt.to_timestamp()

## Missão

Investigue os dados e responda:

1. **Quais unidades consumidoras apresentam queda abrupta de consumo?** Identifique UCs cujo consumo caiu 80-90% de um mês para o outro. Quando isso aconteceu e em qual bairro?
2. **As UCs com queda abrupta estão concentradas geograficamente?** Há um bairro que concentra essas anomalias?
3. **Existe correlação entre as anomalias de consumo e o aumento de ocorrências na rede?** Cruze os dois datasets por bairro e período.
4. **Qual transformador está sobrecarregado?** Identifique transformadores com número de UCs conectadas muito acima de sua capacidade.
5. **O surto de ocorrências está ligado à sobrecarga?** O transformador sobrecarregado é o mesmo associado às ocorrências frequentes?
6. **Estime o prejuízo** da concessionária com o furto: quanto de energia está sendo "desviada" por mês?

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 2. PRÉ-PROCESSAMENTO
# ══════════════════════════════════════════════════════════════════

# Ordena por UC e período para calcular variação mês a mês
df_cs = df_consumo.sort_values(['uc_id', 'ano', 'mes']).copy()

# Consumo do mês anterior (shift dentro de cada UC)
df_cs['consumo_anterior'] = df_cs.groupby('uc_id')['consumo_kwh'].shift(1)

# Variação percentual mês a mês
df_cs['variacao_pct'] = (
    (df_cs['consumo_kwh'] - df_cs['consumo_anterior'])
    / df_cs['consumo_anterior'] * 100
)

# Coluna de período como datetime
df_cs['periodo'] = pd.to_datetime(
    df_cs['ano'].astype(str) + '-' +
    df_cs['mes'].astype(str).str.zfill(2) + '-01'
)

# ── Q1 & Q2: UCs com queda abrupta ≥ 80% ────────────────────────
quedas = df_cs[df_cs['variacao_pct'] <= -80].copy()
quedas['desvio_kwh'] = quedas['consumo_anterior'] - quedas['consumo_kwh']
quedas['periodo'] = pd.to_datetime(
    quedas['ano'].astype(str) + '-' +
    quedas['mes'].astype(str).str.zfill(2) + '-01'
)



### Busca 1 — Existe alguma UC com comportamento anormal de consumo?
O que buscávamos: UCs cujo consumo caiu de forma abrupta de um mês para o outro, indicando adulteração no medidor ou desvio direto na rede.
O que encontramos: 60 UCs apresentaram queda acima de 80% do consumo em um único mês. UCs que consumiam ~200 kWh/mês passaram a registrar menos de 25 kWh. Isso não é variação sazonal nem comportamento normal: é o sinal clássico de um medidor adulterado ou de um ramal clandestino instalado antes do ponto de medição.

In [ ]:
print("=" * 60)
print("Q1 · UCs COM QUEDA ABRUPTA ≥ 80%")
print("=" * 60)
print(f"Total de UCs afetadas : {quedas['uc_id'].nunique()}")

print(f"\nDistribuição por bairro:")
print(quedas['bairro'].value_counts())
print(f"\nDistribuição por período:")
print(quedas.groupby(['ano', 'mes']).size()
      .reset_index(name='n_ucs')
      .sort_values(['ano', 'mes'])
      .to_string(index=False))

print(f"\nTop 10 quedas mais drásticas:")
print(
    quedas[['uc_id', 'bairro', 'ano', 'mes',
            'consumo_anterior', 'consumo_kwh', 'variacao_pct']]
    .sort_values('variacao_pct')
    .head(10)
    .to_string(index=False)
)


### Busca 2 — Essas anomalias estão espalhadas ou concentradas?
*O que buscávamos:* Saber se o problema era difuso (furtos isolados em vários bairros) ou organizado (concentrado em uma região).

*O que encontramos:* 51 das 60 UCs afetadas (85%) estão na Barra do Ceará, e praticamente todas colapsaram no mesmo mês: julho de 2024. Isso elimina a hipótese de furtos individuais espontâneos. A simultaneidade e a concentração geográfica apontam para uma ação coordenada — provavelmente uma rede de desvio organizada no mesmo trecho da distribuição.

In [ ]:
df = pd.read_csv('../data/projeto_06_consumo_energia.csv')
df = df.sort_values(['uc_id', 'ano', 'mes'])

df['data'] = pd.to_datetime(
    df['ano'].astype(str) + '-' + df['mes'].astype(str).str.zfill(2)
)

df['consumo_mes_anterior'] = df.groupby('uc_id')['consumo_kwh'].shift(1)
df['queda_percentual'] = (
    (df['consumo_mes_anterior'] - df['consumo_kwh']) /
    df['consumo_mes_anterior']
) * 100

df_q = df[df['queda_percentual'] >= 80].copy()

por_bairro = (
    df_q.groupby('bairro')
    .size()
    .reset_index(name='total')
    .sort_values('total', ascending=False)
)

# Criação do gráfico de pizza com Matplotlib
BG     = '#0D1117'
PANEL  = '#161B22'
BORDER = '#30363D'
TEXT   = '#E6EDF3'
TEXT_MUTED = '#8B949E'
COLORS = ['#FF4D4D','#FF8C42','#FFD166','#06D6A0','#118AB2',
          '#7B2FBE','#E63946','#F4A261','#2EC4B6','#CBADFF']

bairros  = por_bairro['bairro'].tolist()
cor_map  = {b: COLORS[i % len(COLORS)] for i, b in enumerate(bairros)}
cores_pie = [cor_map[b] for b in bairros]

# Figura
fig, ax = plt.subplots(figsize=(8, 7), facecolor=BG)
ax.set_facecolor(PANEL)

wedges, _, autotexts = ax.pie(
    por_bairro['total'],
    labels=None,
    colors=cores_pie,
    autopct='%1.1f%%',
    startangle=120,
    pctdistance=0.78,
    wedgeprops=dict(width=0.55, edgecolor=BG, linewidth=1.5),
)

for at in autotexts:
    at.set_fontsize(8.5)
    at.set_color(TEXT)

ax.legend(
    handles=[mpatches.Patch(color=cor_map[b], label=f"{b}  ({cor_map[b]})" ) for b in bairros],
    loc='lower center',
    bbox_to_anchor=(0.5, -0.25),
    ncol=2,
    fontsize=8,
    labelcolor=TEXT_MUTED,
    facecolor=PANEL,
    edgecolor=BORDER,
)

ax.set_title(
    'Quedas Abruptas de Consumo (≥ 80%) por Bairro',
    color=TEXT, fontsize=13, fontweight='bold', pad=16
)

plt.tight_layout()
plt.savefig('../docs/pizza_quedas_por_bairro.png', dpi=150,
            bbox_inches='tight', facecolor=BG)
plt.show()

### Busca 3 — As anomalias de consumo coincidem com o aumento de ocorrências na rede?
*O que buscávamos:* Cruzar os dois datasets por bairro e período para ver se o colapso do consumo e o pico de ocorrências aconteceram juntos.

*O que encontramos:* A correlação é direta e visível. As ocorrências em Barra do Ceará saltaram de uma média de 5–10 por mês em 2023 para 22–24 por mês no segundo semestre de 2024, exatamente o período do colapso de consumo. O bairro, que representava menos de 10% das ocorrências da rede no início de 2023, passou a concentrar mais de 40% de todas as ocorrências mensais no final de 2024.


In [ ]:
# ── Q3: Correlação consumo x ocorrências por bairro ─────────────
consumo_bairro = df_cs.groupby(['periodo', 'bairro'])['consumo_kwh'].mean().reset_index()
oc_bairro      = df_oc.groupby(['periodo', 'bairro']).size().reset_index(name='n_oc')

# Merge para cruzamento
correlacao = consumo_bairro.merge(oc_bairro, on=['periodo', 'bairro'], how='left').fillna(0)

print("\n" + "=" * 60)
print("Q3 · CORRELAÇÃO: CONSUMO x OCORRÊNCIAS — BARRA DO CEARÁ")
print("=" * 60)
bdc_corr = correlacao[correlacao['bairro'] == 'Barra do Ceará'].sort_values('periodo')
print(bdc_corr[['periodo', 'consumo_kwh', 'n_oc']].tail(12).to_string(index=False))


In [ ]:
df_consumo = pd.read_csv('../data/projeto_06_consumo_energia.csv')
df_ocorrencias = pd.read_csv('../data/projeto_06_ocorrencias_rede.csv')


df_ocorrencias['data_ocorrencia'] = pd.to_datetime(df_ocorrencias['data_ocorrencia'])


df_ocorrencias['ano'] = df_ocorrencias['data_ocorrencia'].dt.year
df_ocorrencias['mes'] = df_ocorrencias['data_ocorrencia'].dt.month
quedas_por_mes = quedas.groupby(['ano', 'mes', 'bairro']).size().reset_index(name='total_quedas')
ocorrencias_por_mes = df_ocorrencias.groupby(['ano', 'mes', 'bairro']).size().reset_index(name='total_ocorrencias')
df_correlacao = pd.merge(quedas_por_mes, ocorrencias_por_mes, on=['ano', 'mes', 'bairro'], how='outer').fillna(0)
correlacao = df_correlacao['total_quedas'].corr(df_correlacao['total_ocorrencias'])

print(f"O índice de correlação entre quedas de consumo e ocorrências é: {correlacao:.2f}")


In [ ]:


df_mensal = df_correlacao.groupby(['ano', 'mes']).agg({
    'total_quedas': 'sum',
    'total_ocorrencias': 'sum'
}).reset_index()

df_mensal['rotulo'] = df_mensal['mes'].astype(str).str.zfill(2) + '/' + df_mensal['ano'].astype(str)

plt.figure(figsize=(12, 6))

plt.bar(df_mensal['rotulo'], df_mensal['total_ocorrencias'], color='lightgrey', label='Total de Ocorrências (Rede)')

plt.plot(df_mensal['rotulo'], df_mensal['total_quedas'], color='red', marker='o', linewidth=2, label='Quedas de Consumo (>80%)')

plt.title('Correlação Temporal: Ocorrências de Rede vs. Quedas de Consumo', fontsize=14)
plt.xlabel('Mês/Ano', fontsize=12)
plt.ylabel('Quantidade de Eventos', fontsize=12)
plt.legend()
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

### Busca 4 — Qual transformador está sobrecarregado?
*O que buscávamos:* Identificar, no cadastro de transformadores, qual equipamento estava operando muito além da sua capacidade nominal, o que explicaria os apagões recorrentes.

*O que encontramos:* O T-023 (Barra do Ceará) está em colapso: razão de 3,73 UCs por kVA, quase o dobro do segundo mais sobrecarregado e quase 4x acima do limite seguro. Para referência, os demais transformadores operam entre 0,5 e 2,0 UCs/kVA. O próprio cadastro já registra o status do T-023 como "Sobrecarga frequente", o sistema sabia, mas a causa raiz ainda não havia sido identificada.

In [ ]:
# ── Q4: Transformadores sobrecarregados ─────────────────────────
df_transf['ucs_por_kva'] = df_transf['ucs_conectadas'] / df_transf['capacidade_kva']

print("\n" + "=" * 60)
print("Q4 · TRANSFORMADORES SOBRECARREGADOS (top 10)")
print("=" * 60)
print(
    df_transf[['transformador_id', 'bairro', 'capacidade_kva',
                'ucs_conectadas', 'ucs_por_kva', 'status']]
    .sort_values('ucs_por_kva', ascending=False)
    .head(10)
    .to_string(index=False)
)


### Busca 5 — O transformador sobrecarregado é o mesmo responsável pelas ocorrências?
*O que buscávamos:* Confirmar se o T-023 era o epicentro operacional dos problemas, fechando o elo entre furto → sobrecarga → apagões.

*O que encontramos:* Sim, de forma inequívoca. O T-023 acumulou 89 ocorrências no período — 15% de tudo que aconteceu na rede inteira, com apenas 1 transformador de 50. Dessas, 66 foram causadas por Sobrecarga e 26 foram quedas de energia. O segundo transformador mais problemático da rede inteira tem 22 ocorrências no total. O T-023 não é um outlier estatístico, é o centro do problema.

In [ ]:
# ── Q5: Ocorrências por transformador ───────────────────────────
oc_transf = df_oc.groupby('transformador_id').agg(
    total_oc    = ('ocorrencia_id', 'count'),
    quedas_oc   = ('tipo_ocorrencia',  lambda x: (x == 'Queda de energia').sum()),
    sobrecarga  = ('causa_provavel',   lambda x: (x == 'Sobrecarga').sum())
).reset_index()

df_transf_m = df_transf.merge(oc_transf, on='transformador_id', how='left').fillna(0)

print("\n" + "=" * 60)
print("Q5 · OCORRÊNCIAS POR TRANSFORMADOR (top 10)")
print("=" * 60)
print(
    oc_transf.sort_values('total_oc', ascending=False)
    .head(10)
    .to_string(index=False)
)

### Busca 6 — Qual é o prejuízo financeiro?
*O que buscávamos:* Traduzir o desvio de consumo detectado em valor financeiro concreto para a concessionária.

*O que encontramos:* O desvio acumulado estimado é de ~16.743 kWh, resultando em um prejuízo de aproximadamente R$ 15.000 considerando apenas as UCs com queda ≥80% detectada. Este valor é o piso, representa somente o furto flagrante e mensurável. O furto crônico (consumo parcialmente desviado, abaixo do limiar de 80%) não foi capturado por esta análise e provavelmente eleva o prejuízo real em múltiplos desse valor.

In [ ]:
# ── Q6: Estimativa de prejuízo ──────────────────────────────────
total_kwh = quedas['desvio_kwh'].sum()
tarifa_rs  = 0.90  # R$/kWh (tarifa residencial média CE)
prejuizo   = total_kwh * tarifa_rs

print("\n" + "=" * 60)
print("Q6 · ESTIMATIVA DE PREJUÍZO (furto)")
print("=" * 60)
print(f"kWh desviados (total acumulado) : {total_kwh:,.1f} kWh")
print(f"Equivalente em MWh              : {total_kwh/1000:.2f} MWh")
print(f"Prejuízo estimado (R$ 0,90/kWh) : R$ {prejuizo:,.2f}")
print(f"UCs com desvio detectado        : {quedas['uc_id'].nunique()}")
print(f"Desvio médio por UC             : {quedas['desvio_kwh'].mean():.1f} kWh/mês")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 3. VISUALIZAÇÕES
# ══════════════════════════════════════════════════════════════════

BG     = '#0d1117'; PANEL  = '#161b22'; BORDER = '#30363d'
TEXT   = '#e6edf3'; MUTED  = '#8b949e'
RED    = '#ff4d4d'; ORANGE = '#ff9500'; GREEN  = '#3fb950'
BLUE   = '#58a6ff'; YELLOW = '#e3b341'; PURPLE = '#bc8cff'

fig = plt.figure(figsize=(22, 28), facecolor=BG)
fig.subplots_adjust(left=.05, right=.97, top=.95, bottom=.04, hspace=.55, wspace=.35)
gs = gridspec.GridSpec(4, 3, figure=fig)

def style_ax(ax, title, subtitle=''):
    ax.set_facecolor(PANEL)
    for sp in ax.spines.values():
        sp.set_color(BORDER)
        sp.set_linewidth(.8)
    ax.tick_params(colors=MUTED, labelsize=8.5)
    ax.xaxis.label.set_color(MUTED)
    ax.yaxis.label.set_color(MUTED)
    ax.set_title(title, color=TEXT, fontsize=11, fontweight='bold', pad=10, loc='left')
    if subtitle:
        ax.text(0, 1.01, subtitle, transform=ax.transAxes,
                color=MUTED, fontsize=7.5, va='bottom')
    ax.grid(axis='y', color=BORDER, linewidth=.5, linestyle='--', alpha=.6)
    ax.grid(axis='x', visible=False)

fig.text(.5, .972, '⚡  Análise de Perdas Não-Técnicas — Concessionária Ceará',
         ha='center', color=TEXT, fontsize=18, fontweight='bold', fontfamily='monospace')
fig.text(.5, .963,
         'Detecção de furto de energia e sobrecarga de infraestrutura  ·  Jan 2023 – Dez 2024',
         ha='center', color=MUTED, fontsize=10)

# ── Gráfico 1: Série temporal consumo médio por bairro ──────────
ax1 = fig.add_subplot(gs[0, :2])
style_ax(ax1, '1 · Consumo Médio Mensal por Bairro',
         'kWh médio — Barra do Ceará (vermelho) vs demais bairros')

for b in df_consumo['bairro'].unique():
    sub = consumo_bairro[consumo_bairro['bairro'] == b].sort_values('periodo')
    if b == 'Barra do Ceará':
        ax1.plot(sub['periodo'], sub['consumo_kwh'],
                 color=RED, lw=2.5, zorder=5, label='Barra do Ceará')
        ax1.fill_between(sub['periodo'], sub['consumo_kwh'], alpha=.12, color=RED)
    else:
        ax1.plot(sub['periodo'], sub['consumo_kwh'], color=BLUE, lw=1, alpha=.35)

ax1.legend(fontsize=8.5, facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT)
ax1.set_ylabel('Consumo médio (kWh)', color=MUTED, fontsize=9)

# ── Gráfico 2: Quedas por bairro ────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
style_ax(ax2, '2 · UCs c/ Queda ≥80%', 'Por bairro')

qb = quedas['bairro'].value_counts()
colors_b = [RED if b == 'Barra do Ceará' else BLUE for b in qb.index[::-1]]
bars2 = ax2.barh(qb.index[::-1], qb.values[::-1], color=colors_b, height=.65)
for bar, v in zip(bars2, qb.values[::-1]):
    ax2.text(v + .3, bar.get_y() + bar.get_height() / 2,
             str(v), va='center', color=TEXT, fontsize=8.5)
ax2.set_xlabel('Nº de UCs', color=MUTED, fontsize=9)
ax2.tick_params(labelsize=8)

# ── Gráfico 3: Ocorrências mensais BdC vs total ─────────────────
ax3 = fig.add_subplot(gs[1, :2])
style_ax(ax3, '3 · Ocorrências na Rede — Barra do Ceará vs Total',
         'Linha vermelha: Barra do Ceará  ·  Barras azuis: toda a rede')

bdc_oc    = oc_bairro[oc_bairro['bairro'] == 'Barra do Ceará'].sort_values('periodo')
total_oc_t = df_oc.groupby('periodo').size().reset_index(name='n_oc').sort_values('periodo')

ax3b = ax3.twinx()
ax3b.set_facecolor(PANEL)
ax3b.bar(total_oc_t['periodo'], total_oc_t['n_oc'], color=BLUE, alpha=.2, width=20)
ax3b.tick_params(colors=MUTED, labelsize=8)
ax3b.spines['right'].set_color(BORDER)
ax3b.set_ylabel('Total rede', color=MUTED, fontsize=9)

ax3.plot(bdc_oc['periodo'], bdc_oc['n_oc'],
         color=RED, lw=2.5, marker='o', ms=5, zorder=5, label='Barra do Ceará')
ax3.fill_between(bdc_oc['periodo'], bdc_oc['n_oc'], alpha=.15, color=RED)
ax3.axvline(pd.Timestamp('2024-07-01'), color=ORANGE, lw=1.4, ls='--', alpha=.85)
ax3.set_ylabel('Ocorrências — BdC', color=MUTED, fontsize=9)

l1, lb1 = ax3.get_legend_handles_labels()
l2, lb2 = ax3b.get_legend_handles_labels()
ax3.legend(l1 + l2, lb1 + lb2, fontsize=8, facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT)

# ── Gráfico 4: Tipos de ocorrência BdC ─────────────────────────
ax4 = fig.add_subplot(gs[1, 2])
style_ax(ax4, '4 · Tipos de Ocorrência', 'Barra do Ceará')

bdc_tipos = df_oc[df_oc['bairro'] == 'Barra do Ceará']['tipo_ocorrencia'].value_counts()
pal4  = [RED, ORANGE, YELLOW, BLUE, GREEN, PURPLE]
bars4 = ax4.barh(bdc_tipos.index[::-1], bdc_tipos.values[::-1],
                 color=pal4[:len(bdc_tipos)][::-1], height=.65)
for bar, v in zip(bars4, bdc_tipos.values[::-1]):
    ax4.text(v + .3, bar.get_y() + bar.get_height() / 2,
             str(v), va='center', color=TEXT, fontsize=8)
ax4.set_xlabel('Ocorrências', color=MUTED, fontsize=9)
ax4.tick_params(labelsize=7.5)

# ── Gráfico 5: Sobrecarga de transformadores ────────────────────
ax5 = fig.add_subplot(gs[2, :2])
style_ax(ax5, '5 · Sobrecarga de Transformadores',
         'UCs conectadas / kVA — linha verde = limite seguro')

top15    = df_transf.sort_values('ucs_por_kva', ascending=False).head(15)
colors5  = [
    RED    if r['transformador_id'] == 'T-023' else
    ORANGE if r['ucs_por_kva'] > 1.5 else BLUE
    for _, r in top15.iterrows()
]
bars5 = ax5.bar(top15['transformador_id'], top15['ucs_por_kva'],
                color=colors5, width=.6)
ax5.axhline(1.0, color=GREEN, lw=1.4, ls='--', alpha=.8, label='Limite seguro')
for bar, (_, r) in zip(bars5, top15.iterrows()):
    ax5.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + .04,
             r['bairro'][:8], ha='center', color=MUTED, fontsize=6.5, rotation=30)
ax5.set_ylabel('UCs / kVA', color=MUTED, fontsize=9)
ax5.tick_params(axis='x', rotation=45, labelsize=8)
ax5.legend(fontsize=8, facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT)

ids = list(top15['transformador_id'])
if 'T-023' in ids:
    idx = ids.index('T-023')
    ax5.annotate('T-023 ⚠\nBarra do Ceará\nSobrecarga crítica',
                 xy=(idx, top15.iloc[idx]['ucs_por_kva']),
                 xytext=(idx + 2, top15.iloc[idx]['ucs_por_kva'] - .6),
                 color=RED, fontsize=8.5, fontweight='bold',
                 arrowprops=dict(arrowstyle='->', color=RED, lw=1.5))

# ── Gráfico 6: T-023 ocorrências vs rede ───────────────────────
ax6 = fig.add_subplot(gs[2, 2])
style_ax(ax6, '6 · T-023 vs Rede',
         'Ocorrências mensais — linha vermelha vs barras azuis (outros)')

t023_m  = df_oc[df_oc['transformador_id'] == 'T-023'].groupby('periodo').size()
outros_m = df_oc[df_oc['transformador_id'] != 'T-023'].groupby('periodo').size()

ax6b = ax6.twinx()
ax6b.bar(outros_m.index, outros_m.values, color=BLUE, alpha=.2, width=20)
ax6b.tick_params(colors=MUTED, labelsize=8)
ax6b.spines['right'].set_color(BORDER)
ax6b.set_ylabel('Outros trafo', color=MUTED, fontsize=8)

ax6.plot(t023_m.index, t023_m.values,
         color=RED, lw=2.5, marker='o', ms=4, zorder=5)
ax6.fill_between(t023_m.index, t023_m.values, alpha=.15, color=RED)
ax6.set_ylabel('T-023', color=MUTED, fontsize=9)
ax6.tick_params(axis='x', rotation=45, labelsize=7)


# ── Gráfico 7: Heatmap causa × bairro ──────────────────────────
ax7 = fig.add_subplot(gs[3, :2])
style_ax(ax7, '7 · Heatmap: Causa × Bairro', 'Número de ocorrências')

heat     = df_oc.groupby(['bairro', 'causa_provavel']).size().unstack(fill_value=0)
causas   = heat.columns.tolist()
bairros_h = heat.index.tolist()
data_h   = heat.values
vmax     = data_h.max()
cmap     = plt.cm.Blues

for i, b in enumerate(bairros_h):
    for j, c in enumerate(causas):
        v   = data_h[i, j]
        col = cmap(v / vmax) if vmax > 0 else (0, 0, 0, 1)
        ax7.add_patch(plt.Rectangle([j - .5, i - .5], 1, 1,
                                    facecolor=col, edgecolor=PANEL, lw=.5))
        ax7.text(j, i, str(v), ha='center', va='center',
                 color=TEXT if v / vmax > .5 else MUTED,
                 fontsize=9, fontweight='bold')

ax7.set_xlim(-.5, len(causas) - .5)
ax7.set_ylim(-.5, len(bairros_h) - .5)
ax7.set_xticks(range(len(causas)))
ax7.set_xticklabels(causas, rotation=20, ha='right', color=MUTED, fontsize=9)
ax7.set_yticks(range(len(bairros_h)))
ax7.set_yticklabels(bairros_h, color=MUTED, fontsize=9)
for sp in ax7.spines.values():
    sp.set_visible(False)
ax7.grid(visible=False)
if 'Barra do Ceará' in bairros_h:
    bi = bairros_h.index('Barra do Ceará')
    ax7.add_patch(plt.Rectangle([-.5, bi - .5], len(causas), 1,
                                fill=False, edgecolor=RED, lw=2.2, zorder=10))


# ── Gráfico 8: Prejuízo estimado por mês ───────────────────────
ax8 = fig.add_subplot(gs[3, 2])
style_ax(ax8, '8 · Prejuízo Estimado',
         'MWh desviados por mês (UCs com queda ≥80%)')

desvio_mes = (quedas.groupby('periodo')['desvio_kwh']
              .sum().reset_index().sort_values('periodo'))
ax8.bar(desvio_mes['periodo'], desvio_mes['desvio_kwh'] / 1000,
        color=RED, width=20, alpha=.85)
ax8.set_ylabel('MWh desviados', color=MUTED, fontsize=9)
ax8.tick_params(axis='x', rotation=45, labelsize=7.5)

fig.text(.97, .046,
         f'💸 Total desviado : {total_kwh / 1000:.1f} MWh\n'
         f'   Prejuízo est.  : R$ {prejuizo:,.0f}',
         ha='right', va='bottom',
         color=YELLOW, fontsize=9.5, fontweight='bold', fontfamily='monospace',
         bbox=dict(boxstyle='round,pad=.6', facecolor=PANEL, edgecolor=YELLOW, lw=1.3))

plt.savefig('../docs/analise_perdas_energia.png', dpi=160, bbox_inches='tight', facecolor=BG)
print("✅ Análise concluída. Gráfico salvo em 'docs/analise_perdas_energia.png'")


## Conclusão

A análise dos dados confirmou a hipótese inicial da engenharia: o bairro Barra do Ceará concentra o principal problema da rede, tendo o transformador T-023 como ponto crítico. O evento registrado em julho de 2024, no qual 51 unidades consumidoras foram afetadas simultaneamente, sugere a ocorrência de furto de energia de forma organizada, contribuindo diretamente para a sobrecarga do transformador e para os apagões recorrentes na região.

Entretanto, o problema não pode ser explicado apenas por essa causa. Uma hipótese complementar é que a infraestrutura elétrica do bairro não acompanhou o crescimento urbano, fazendo com que a rede opere próxima ou acima de sua capacidade. Assim, os apagões podem resultar da combinação entre furtos de energia e limitações estruturais da rede.

Diante desse cenário, a solução exige duas frentes de atuação: ações de fiscalização para identificar e combater irregularidades e investimentos na expansão da infraestrutura elétrica, com instalação de novos transformadores e melhor planejamento da distribuição de carga no bairro.